In [ ]:
import pandas as pd

df = pd.read_csv("order.csv")

print(df)
df.head()
df.shape
df.columns
df.info()


In [ ]:
df.isnull().sum()

In [ ]:
# Duplicate records
df.duplicated().sum()

df[df.duplicated()]


In [ ]:
# Invalid values check

df[df["quantity"] <= 0]


df[df["price"] <= 0]

#Invalid status:

df[~df["status"].isin(["Delivered", "Cancelled"])]

In [ ]:
#Missing numeric values:

df["quantity"] = df["quantity"].fillna(df["quantity"].median())
df["price"] = df["price"].fillna(df["price"].median())

#Missing categorical values:

df["customer"] = df["customer"].fillna("Unknown")
df["category"] = df["category"].fillna("Unknown")
df["status"] = df["status"].fillna("Unknown")

In [ ]:
#Duplicates remove:

df = df.drop_duplicates()

In [ ]:
#Invalid quantity remove:

df = df[df["quantity"] > 0]

#Invalid price remove:

df = df[df["price"] > 0]

In [ ]:
#Create Revenue
df["revenue"] = df["quantity"] * df["price"]

In [ ]:
# Category-level metrics
category_summary = df.groupby("category").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    avg_revenue=("revenue", "mean")
)

print(category_summary)

In [ ]:
#Cancellation rate
cancel_rate = (
    df.groupby("category")["status"]
    .apply(lambda x: (x == "Cancelled").mean() * 100)
)

print(cancel_rate)

In [ ]:
#Cancellation rate
cancel_rate = (
    df.groupby("category")["status"]
    .apply(lambda x: (x == "Cancelled").mean() * 100)
)

print(cancel_rate)

In [ ]:
# Combine metrics
category_summary["cancel_rate"] = cancel_rate

In [ ]:
# Highest-risk category

category_summary["risk_score"] = (
    category_summary["cancel_rate"] * 0.6
    + (1 / category_summary["total_revenue"]) * 0.4
)

In [ ]:
#rank
category_summary = category_summary.sort_values(
    "risk_score",
    ascending=False
)

print(category_summary)

In [ ]:
# final 
highest_risk = category_summary.index[0]

print("Category needing attention first:", highest_risk)